# Steam Recommender Modeling

Baseline comparison and matrix factorization experiments.

In [ ]:
# Steam Recommender Modeling
# Baseline comparison and matrix factorization experiments.

# ── Cell 1: Imports ────────────────────────────────────────────────────────────
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.baselines import evaluate_recommendations, popularity_recommendations, random_recommendations
from src.data_loader import build_interaction_matrices, leave_one_out_split
from src.matrix_factorization import MatrixFactorizationSGD, build_test_items_by_user, evaluate_mf_leave_one_out


# ── Cell 2: Load & split data ──────────────────────────────────────────────────
DATA_DIR = ROOT / 'data'
interactions = pd.read_csv(DATA_DIR / 'interactions_filtered.csv')
train_df, test_df = leave_one_out_split(interactions)

print('Train rows:', len(train_df))
print('Test rows:', len(test_df))


# ── Cell 3: Baselines ──────────────────────────────────────────────────────────
train_data = build_interaction_matrices(train_df)
user_ids = sorted(test_df['user_id'].unique().tolist())
all_game_ids = sorted(train_df['app_id'].unique().tolist())
test_items_by_user_id = test_df.groupby('user_id')['app_id'].apply(list).to_dict()

k_eval = 10
results = []

pop_recs = popularity_recommendations(train_df, user_ids=user_ids, k=k_eval)
pop_scores = evaluate_recommendations(pop_recs, test_items_by_user_id, k=k_eval)
results.append({'model': 'Popularity', 'precision@10': pop_scores['precision_at_k'], 'recall@10': pop_scores['recall_at_k']})

rand_recs = random_recommendations(train_df, user_ids=user_ids, all_game_ids=all_game_ids, k=k_eval, seed=42)
rand_scores = evaluate_recommendations(rand_recs, test_items_by_user_id, k=k_eval)
results.append({'model': 'Random', 'precision@10': rand_scores['precision_at_k'], 'recall@10': rand_scores['recall_at_k']})


# ── Cell 4: Matrix factorization ───────────────────────────────────────────────
test_items_by_user_idx = build_test_items_by_user(test_df, train_data.user_to_idx, train_data.game_to_idx)

for latent_k in [20, 50, 100]:
    model = MatrixFactorizationSGD(k=latent_k, reg=0.01, learning_rate=0.01, epochs=20, random_state=42)

    # FIX A: Train on hours_matrix (log1p playtime) — richer signal than binary positive.
    # FIX B: Pass the same matrix (hours_matrix) to evaluate so "seen" item masking
    #         is consistent with what was trained on.
    model.fit(train_data.hours_matrix, verbose=True)
    mf_scores = evaluate_mf_leave_one_out(
        model,
        train_data.hours_matrix,   # FIX B: was binary_matrix — now consistent with training
        test_items_by_user_idx,
        k=k_eval,
    )
    results.append(
        {
            'model': f'MF (k={latent_k})',
            'precision@10': mf_scores['precision_at_k'],
            'recall@10': mf_scores['recall_at_k'],
        }
    )


# ── Cell 5: Results ────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).sort_values('precision@10', ascending=False).reset_index(drop=True)
results_df

Train rows: 20167480
Test rows: 1906301


In [ ]:
DATA_DIR = ROOT / 'data'

interactions = pd.read_csv(DATA_DIR / 'interactions_filtered.csv')

train_df, test_df = leave_one_out_split(interactions)



print('Train rows:', len(train_df))

print('Test rows:', len(test_df))

In [ ]:
train_data = build_interaction_matrices(train_df)

user_ids = sorted(test_df['user_id'].unique().tolist())

all_game_ids = sorted(train_df['app_id'].unique().tolist())

test_items_by_user_id = test_df.groupby('user_id')['app_id'].apply(list).to_dict()



k_eval = 10

results = []



pop_recs = popularity_recommendations(train_df, user_ids=user_ids, k=k_eval)

pop_scores = evaluate_recommendations(pop_recs, test_items_by_user_id, k=k_eval)

results.append({'model': 'Popularity', 'precision@10': pop_scores['precision_at_k'], 'recall@10': pop_scores['recall_at_k']})



rand_recs = random_recommendations(train_df, user_ids=user_ids, all_game_ids=all_game_ids, k=k_eval, seed=42)

rand_scores = evaluate_recommendations(rand_recs, test_items_by_user_id, k=k_eval)

results.append({'model': 'Random', 'precision@10': rand_scores['precision_at_k'], 'recall@10': rand_scores['recall_at_k']})

In [ ]:
test_items_by_user_idx = build_test_items_by_user(test_df, train_data.user_to_idx, train_data.game_to_idx)

for latent_k in [20, 50, 100]:
    model = MatrixFactorizationSGD(k=latent_k, reg=0.01, learning_rate=0.01, epochs=20, random_state=42)
    model.fit(train_data.positive_matrix, verbose=True)
    mf_scores = evaluate_mf_leave_one_out(model, train_data.binary_matrix, test_items_by_user_idx, k=k_eval)
    results.append(
        {
            'model': f'MF (k={latent_k})',
            'precision@10': mf_scores['precision_at_k'],
            'recall@10': mf_scores['recall_at_k'],
        }
    )

In [ ]:
results_df = pd.DataFrame(results).sort_values('precision@10', ascending=False).reset_index(drop=True)

results_df